## 2. Veri Ön İşleme

Bu notebook içerisinde, Autoflow-Sync akıllı lojistik projemizin ilk aşaması olan Veri Keşfi (EDA) dökümanından elde ettiğimiz yapısal tahliller doğrultusunda ham verimizi temizliyoruz. Yapay zeka modellerinin (XGBoost, Random Forest vb.) veriyi hatasız işleyebilmesi ve en yüksek doğrulukla tahmin üretebilmesi için ham değişkenleri optimize hale getireceğiz.

Amaçlarımız:


Eksik Verilerin Yönetimi: Mantıksal yaklaşımlarla eksik verileri sisteme kazandırmak.

Feature Engineering: Tarihsel döngüleri modelin öğrenebileceği parçalara ayırmak.

Kategorik Verilerin Dönüştürülmesi (Encoding): Metinsel lojistik verilerini sayısal ifadelere çevirmek.

Veri Setinin Temizlenmesi: Tahminleme sürecine katkısı olmayan değişkenleri eleyerek nihai veri setini kaydetmek.

In [1]:
import pandas as pd
import numpy as np
import os

# Veri setinin projedeki konumunu dinamik olarak tanımlıyoruz
data_path = os.path.join("..", "..", "..", "data", "raw", "smart_logistics_dataset.csv")

# Ham veriyi DataFrame olarak hafızaya alıyoruz
df = pd.read_csv(data_path)

# Orijinal veriyi korumak adına ön işleme adımları için bir kopya oluşturuyoruz
df_processed = df.copy()

print("Sistem Bilgisi: Ön işleme adımları için ham veri başarıyla kopyalandı.")

Sistem Bilgisi: Ön işleme adımları için ham veri başarıyla kopyalandı.


Ön işleme adımlarında ham verinin yapısını bozmamak ve olası bir geriye dönük kontrol ihtiyacında orijinal veriye sadık kalabilmek adına `df_processed` adında güvenli bir çalışma kopyası oluşturduk. Bu adımdan itibaren tüm yapısal değişiklikler bu kopya üzerinden yürütülecektir.

## 2.1. Eksik Verilerin Yönetimi

Veri keşfi (EDA) aşamasında missingno matrisinde gördüğümüz üzere, veri setimizde yalnızca `Logistics_Delay_Reason` (Gecikme Nedeni) sütununda eksik veriler bulunuyordu. 

Lojistik operasyonun doğası gereği, zamanında tamamlanan sevkiyatlarda (`Logistics_Delay` = 0) bir gecikme nedeni girilmediği için bu alanlar boş kalmıştır. Veri kaybını önlemek ve bu mantıksal boşluğu modele doğru öğretmek adına ilgili satırları silmek yerine 'No Delay' (Gecikme Yok) ifadesiyle dolduruyoruz.

In [2]:
# Sütundaki boş (NaN) değerleri 'No Delay' sabitiyle dolduruyoruz
df_processed['Logistics_Delay_Reason'] = df_processed['Logistics_Delay_Reason'].fillna('No Delay')

# İşlemin başarısını doğrulamak için sütunun güncel dağılımını kontrol ediyoruz
print("Logistics_Delay_Reason Sütununun Yeni Dağılımı:")
print(df_processed['Logistics_Delay_Reason'].value_counts())

Logistics_Delay_Reason Sütununun Yeni Dağılımı:
Logistics_Delay_Reason
Weather               267
No Delay              263
Traffic               236
Mechanical Failure    234
Name: count, dtype: int64


Yapılan eksik veri dolgusu (imputation) sonrasında, satır kaybı yaşanmadan veri setinin 1000 satırlık bütünlüğü korunmuştur. Eskiden boş olan 263 satır, operasyonel mantığa uygun olarak 'No Delay' kategorisine dönüştürülmüştür. Böylece makine öğrenmesi algoritmasının zamanında ulaşan sevkiyatların karakteristik özelliklerini bu etiket üzerinden de anlamlandırması sağlanmıştır.

## 2.2. Zaman Serisi ve Özellik Mühendisliği (Feature Engineering)
Otomotiv lojistiğinde zaman faktörü mevsimsel talepleri, vardiya değişimlerini ve trafik yoğunluklarını doğrudan etkiler. Bu aşamada, metinsel `Timestamp` sütununu tarih-saat formatına çevirerek modelimizin zaman tabanlı örüntüleri yakalayabilmesi adına 3 yeni anlamlı öznitelik (feature) üretiyoruz[cite: 2]:

1. **Month (Ay):** Mevsimsel etkileri ve dönemlik yoğunlukları yakalamak için[cite: 2].
2. **Day_of_Week (Haftanın Günü):** Hafta içi ve hafta sonu lojistik dinamiklerinin farkını görmek için[cite: 2].
3. **Hour (Saat):** Gün içindeki vardiya geçişlerini ve pik trafik saatlerini anlamlandırmak için[cite: 2].

*Not: Hücrenin arka arkaya çalıştırılması durumunda `KeyError` hatası almamak adına, 'Timestamp' sütunu kontrol edilerek işlem yapılacak ve ardından drop edilecektir.*

In [8]:
# 'Timestamp' sütununun veri setinde var olup olmadığını kontrol ediyoruz (Tekrar çalıştırma hatasını önler)
if 'Timestamp' in df_processed.columns:
    # Metin formatındaki Timestamp sütununu datetime nesnesine çeviriyoruz
    df_processed['Timestamp'] = pd.to_datetime(df_processed['Timestamp'])

    # Yeni özellikleri (features) türetiyoruz
    df_processed['Month'] = df_processed['Timestamp'].dt.month
    df_processed['Day_of_Week'] = df_processed['Timestamp'].dt.dayofweek  # 0=Pazartesi, 6=Pazar
    df_processed['Hour'] = df_processed['Timestamp'].dt.hour

    # Bilgileri çıkardığımıza göre artık ham Timestamp sütununu güvenle siliyoruz
    df_processed = df_processed.drop(columns=['Timestamp'])
    print("Sistem Bilgisi: 'Timestamp' sütunu başarıyla parçalandı ve ham sütun elendi.")
else:
    print("Sistem Bilgisi: 'Timestamp' sütunu zaten daha önce elenmiş. Adım güvenle atlanıyor.")

# Kontrol için zaman özelliklerinin güncel durumuna göz atıyoruz
print("\nMevcut Zaman Özellikleri (İlk 3 Satır):")
print(df_processed[['Month', 'Day_of_Week', 'Hour']].head(3))

Sistem Bilgisi: 'Timestamp' sütunu zaten daha önce elenmiş. Adım güvenle atlanıyor.

Mevcut Zaman Özellikleri (İlk 3 Satır):
   Month  Day_of_Week  Hour
0      3            2     0
1     10            2     7
2      7            0    18


### Zaman Özellikleri Analiz Yorumu
Yaptığımız Özellik Mühendisliği (Feature Engineering) çalışmasıyla, tek bir metin sütunundan modelin doğrusal olarak öğrenebileceği 3 yeni stratejik boyut kazandırdık[cite: 2]. Ham `Timestamp` sütunu elenerek veri seti sadeleştirilmiştir[cite: 2]. Hücreye eklediğimiz `if` kontrolü sayesinde, notebook'un tekrar çalıştırılması durumunda oluşabilecek `KeyError` riskleri tamamen engellenmiştir. Bu sayede algoritma, Cuma günleri akşam vardiyasındaki (`Day_of_Week`=4, `Hour`=17) yoğunlukların sevkiyat gecikmelerine olan etkisini kararlı bir şekilde hesaplayabilecektir[cite: 2].

## 2.3. Kategorik Değişkenlerin Dönüştürülmesi (Encoding)
Makine öğrenmesi algoritmaları metinsel verileri doğrudan işleyemez; tüm girdilerin sayısal olmasını bekler. Veri keşfi aşamasında tespit ettiğimiz kategorik sütunlarımızı (`Asset_ID`, `Shipment_Status`, `Traffic_Status`, `Logistics_Delay_Reason`) uygun yöntemlerle sayısal değerlere dönüştüreceğiz[cite: 2]. 

Burada veri sızıntısını (data leakage) önlemek ve kurumsal standartları korumak adına `pandas.get_dummies` yerine `scikit-learn` kütüphanesinin dönüşüm araçlarını tercih ediyoruz[cite: 2].

In [9]:
from sklearn.preprocessing import LabelEncoder

# Dönüştürülecek kategorik sütunları bir liste halinde tanımlıyoruz
categorical_cols = ['Asset_ID', 'Shipment_Status', 'Traffic_Status', 'Logistics_Delay_Reason']

# scikit-learn LabelEncoder nesnesini başlatıyoruz
le = LabelEncoder()

# Her bir kategorik sütun üzerinde döngü kurarak sayısal dönüşümü uyguluyoruz
for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])

# Kontrol amacıyla dönüşüm sonrası veri setinin ilk 3 satırını görüntülüyoruz
print("Dönüşüm Sonrası Sayısal Hale Gelen Kategorik Sütunlar (İlk 3 Satır):")
print(df_processed[categorical_cols].head(3))

Dönüşüm Sonrası Sayısal Hale Gelen Kategorik Sütunlar (İlk 3 Satır):
   Asset_ID  Shipment_Status  Traffic_Status  Logistics_Delay_Reason
0         7                0               1                       1
1         6                2               2                       3
2         1                2               1                       1


### Kategorik Dönüşüm Analiz Yorumu
Bu kod bloğu ile veri setindeki tüm metinsel (kategorik) lojistik değişkenleri, modelleme algoritmalarının matematiksel olarak işleyebileceği sayısal değerlere (0, 1, 2 vb.) dönüştürdük[cite: 2]. Dönüşüm esnasında endüstriyel standart olan `scikit-learn` kütüphanesini kullanarak veri sızıntısının (data leakage) önüne geçtik ve veri kalitemizi koruduk[cite: 2]. Örneğin, `Traffic_Status` içerisindeki 'Detour', 'Heavy' ve 'Clear' gibi sözel durumlar artık yapay zekanın doğrudan korelasyon kurabileceği kararlı birer sayısal öznitelik haline gelmiştir[cite: 2].

## 2.4. Modelleme Öncesi Gereksiz Sütunların Elenmesi
Veri ön işleme sürecinin bu adımında, makine öğrenmesi modelinin gecikme tahminleme başarısına doğrudan bir katkısı bulunmayan, aksine veri setinde gürültü (noise) yaratarak modelin ezberlemesine (overfitting) yol açabilecek sütunları eliyoruz. 

Bu doğrultuda, satırların benzersiz kaydı olan `index` sütununu, araçların koordinatlarını belirten ancak rota optimizasyonu yerine doğrudan gecikme tahmini yapacağımız için bu aşamada gürültü yaratabilecek `Latitude` ve `Longitude` sütunlarını temizliyoruz[cite: 2]. Ayrıca müşteri bazlı olan `User_Purchase_Frequency` sütununu da lojistik süreç odaklı çalıştığımız için veri etiketlerimiz arasından çıkarıyoruz[cite: 2].

In [11]:
# Belirlediğimiz gereksiz ve gürültü yaratan sütunları eliyoruz
columns_to_drop = ['index', 'Latitude', 'Longitude', 'User_Purchase_Frequency']
existing_columns_to_drop = [col for col in columns_to_drop if col in df_processed.columns]
if existing_columns_to_drop:
    df_processed = df_processed.drop(columns=existing_columns_to_drop)
    print(f"Silinen sütunlar: {existing_columns_to_drop}")
else:
    print("Silinecek sütun bulunamadı; veri seti olduğu gibi kaldı.")

# Kalan sütunları kontrol ediyoruz
print("Veri Setinde Kalan Sütunlar:")
print(df_processed.columns.tolist())

Silinen sütunlar: ['Latitude', 'Longitude', 'User_Purchase_Frequency']
Veri Setinde Kalan Sütunlar:
['Asset_ID', 'Inventory_Level', 'Shipment_Status', 'Temperature', 'Humidity', 'Traffic_Status', 'Waiting_Time', 'User_Transaction_Amount', 'Logistics_Delay_Reason', 'Asset_Utilization', 'Demand_Forecast', 'Logistics_Delay', 'Month', 'Day_of_Week', 'Hour']


## 2.5. Temizlenmiş Nihai Veri Setinin Kaydedilmesi
Tüm temizleme, özellik mühendisliği (feature engineering) ve encoding işlemlerini başarıyla tamamladığımız verimizi, projenin kurumsal klasör yapısına uygun olarak `data/processed/` dizini altına `processed_smart_logistics_dataset.csv` ismiyle kaydediyoruz[cite: 2]. Bu veri seti, bir sonraki aşamada makine öğrenmesi modellerimizin eğitilmesinde doğrudan kullanılacaktır[cite: 2].

In [12]:
# Klasör yolunu dinamik olarak tanımlıyoruz
output_dir = os.path.join("..", "..", "..", "data", "processed")

# Eğer processed klasörü yoksa otomatik oluşturulmasını sağlıyoruz
os.makedirs(output_dir, exist_ok=True)

# Temizlenmiş veriyi CSV olarak kaydediyoruz (indeks sütununu eklemiyoruz)
output_path = os.path.join(output_dir, "processed_smart_logistics_dataset.csv")
df_processed.to_csv(output_path, index=False)

print(f"Sistem Bilgisi: Temizlenmiş veri seti başarıyla kaydedildi -> {output_path}")
print(f"Güncel Veri Boyutu: {df_processed.shape[0]} satır, {df_processed.shape[1]} sütun.")

Sistem Bilgisi: Temizlenmiş veri seti başarıyla kaydedildi -> ..\..\..\data\processed\processed_smart_logistics_dataset.csv
Güncel Veri Boyutu: 1000 satır, 15 sütun.
